In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
import numpy as np
import os

In [2]:
MODEL_PATH= 'EP_models/'
os.environ['HF_HOME']= MODEL_PATH  # before import transformers
os.environ['HF_DATASETS_CACHE']= MODEL_PATH  # dataset cache

import torch
import transformers
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_gif
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# filter warnings
import warnings
transformers.logging.set_verbosity_error()
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

print(f'PyTorch version= {torch.__version__}')
print(f'transformers version= {transformers.__version__}')
print(f'CUDA available= {torch.cuda.is_available()}')

Device = 'cuda:0'
Dtype = torch.float16

PyTorch version= 2.10.0
transformers version= 5.0.0
CUDA available= False


In [3]:
Step = 4  # options: [1,2,4,8]
Repo = 'ByteDance/AnimateDiff-Lightning'
chkpt = f'animatediff_lightning_{Step}step_diffusers.safetensors'
Base = 'emilianJR/epiCRealism'  # pick a base model

adapter = MotionAdapter().to(Device, Dtype)
adapter.load_state_dict(load_file(hf_hub_download(Repo, chkpt), device=Device))
Pipe_dif = AnimateDiffPipeline.from_pretrained(Base, motion_adapter=adapter, torch_dtype=Dtype).to(Device)
Pipe_dif.scheduler = EulerDiscreteScheduler.from_config(Pipe_dif.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
output = Pipe_dif(prompt='a girl smiling', guidance_scale=1.0, num_inference_steps=Step)
export_to_gif(output.frames[0], 'animation1.gif')

In [ ]:
output = Pipe_dif(prompt='a rocket taking off', guidance_scale=1.0, num_inference_steps=Step)
export_to_gif(output.frames[0], 'animation2.gif')

## Example Video Generation

In [ ]:
from diffusers import DiffusionPipeline
from PIL import Image

Pipe_dif = DiffusionPipeline.from_pretrained('stabilityai/stable-video-diffusion-img2vid').to(Device)

In [ ]:
# load sample input image and resize it -- user provided
input_image = Image.open("module06_example_512.jpg")
input_image = input_image.resize((256, 256))

# generate video frames from the image
video_frames = Pipe_dif(input_image, num_frames=12).frames

In [ ]:
# save each frame as an image, or combine them to make a video
for i, frame in enumerate(video_frames[0]):
    frame.save(f'frame_{i:03d}.png')

In [ ]:
import cv2

print(cv2.__version__)

img_folder = '.'  # current folder
vid_name = 'output.mp4'

images = [img for img in os.listdir(img_folder) if img.endswith('.png')]
images.sort()  # ensure correct order

frame = cv2.imread(os.path.join(img_folder, images[0]))
height, width, layers = frame.shape

fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # or 'XVID' for .avi
video = cv2.VideoWriter(vid_name, fourcc, 10, (width, height))  # 10 frames per sec

for image in images:
    img = cv2.imread(os.path.join(img_folder, image))
    video.write(img)

video.release()
cv2.destroyAllWindows()

print(f'video created: {vid_name}')